# Stage 3: Baseline CNN
**Deep Learning-Based Detection and Classification of Brain Tumors Using MRI Images**

A small CNN trained **from scratch** (no pretrained weights). It is the reference that the ResNet models must beat in Stages 4 and 5.

Rules followed here: the model is selected using **validation** data only (macro F1), the test set is evaluated once after training,
and every number printed comes from the code below.

Experiments in this notebook
* **B1**: plain cross-entropy (main baseline)
* **B2** (optional): class-weighted cross-entropy, a controlled comparison to answer "does the moderate class imbalance matter?"

Run on a GPU runtime. Run the cells in order.

## 1. Folders and source modules

In [ ]:
from pathlib import Path
PROJECT = Path('/content/brain-tumor-project')
for d in ['data','notebooks','src','models','results/figures','results/metrics','results/predictions']:
    (PROJECT/d).mkdir(parents=True, exist_ok=True)

In [ ]:
%%writefile /content/brain-tumor-project/src/utils.py
"""Reproducibility helpers."""
import os, random, platform
import numpy as np


def set_seed(seed=42):
    """Seed Python, NumPy, PyTorch and CUDA. Deterministic cuDNN is requested (can be slightly slower)."""
    import torch
    random.seed(seed); np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def env_info(seed=42):
    import torch, torchvision
    return dict(python=platform.python_version(), torch=torch.__version__, torchvision=torchvision.__version__,
                cuda_available=torch.cuda.is_available(),
                gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, seed=seed)

In [ ]:
%%writefile /content/brain-tumor-project/src/data_prep.py
"""Dataset inventory, duplicate handling, grouping and splitting.

This module does not need PyTorch. All numbers it reports are computed from
the files on disk. The folder layout it expects was verified in Stage 1:
    <root>/{Train,Val}/<ClassName>/{images,labels}/<file>
and it raises a clear error if the layout is different.
"""
import hashlib, json
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image

CLASS_NAMES = ['Glioma', 'Meningioma', 'No Tumor', 'Pituitary']   # alphabetical, same order as the box class ids
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}


def find_dataset_root(base):
    """Return the folder that directly contains the Train and Val folders."""
    base = Path(base)
    for p in [base] + [q for q in base.rglob('*') if q.is_dir()]:
        names = {c.name.lower() for c in p.iterdir() if c.is_dir()}
        if {'train', 'val'} <= names:
            return p
    raise FileNotFoundError(f'No folder containing Train/ and Val/ found under {base}')


def _dhash(img):
    a = np.asarray(img.convert('L').resize((9, 8), Image.LANCZOS), dtype=np.int16)
    bits = (a[:, 1:] > a[:, :-1]).flatten()
    return int(np.packbits(bits).view('>u8')[0])


def _md5(path):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()


def _read_boxes(txt_path):
    """YOLO lines: class cx cy w h (normalised). Returns a list of lists."""
    if txt_path is None:
        return []
    boxes = []
    for line in txt_path.read_text().strip().splitlines():
        t = line.split()
        if len(t) >= 5:
            boxes.append([int(float(t[0]))] + [float(v) for v in t[1:5]])
    return boxes


def build_inventory(root, progress=lambda x, **k: x):
    """One row per image: path, class, original split, size, hashes, boxes."""
    root = Path(root)
    label_files = {p.stem: p for p in root.rglob('*.txt')}
    paths = sorted(p for p in root.rglob('*') if p.is_file() and p.suffix.lower() in IMG_EXT)
    rows, unreadable = [], []
    for p in progress(paths, desc='inventory'):
        rel = p.relative_to(root)
        if len(rel.parts) < 3 or rel.parts[1] not in CLASS_TO_IDX:
            raise ValueError(f'Unexpected path layout: {rel}. Expected <Split>/<Class>/images/<file>.')
        try:
            with Image.open(p) as im:
                im.verify()
            with Image.open(p) as im:
                w, h = im.size
                mode, fmt = im.mode, im.format
                dh = _dhash(im)
                rgb = np.asarray(im.convert('RGB'))
        except Exception as e:
            unreadable.append((str(rel), repr(e)))
            continue
        boxes = _read_boxes(label_files.get(p.stem))
        rows.append(dict(
            rel=str(rel), split_orig=rel.parts[0].lower(), label=rel.parts[1],
            label_idx=CLASS_TO_IDX[rel.parts[1]], stem=p.stem, width=w, height=h, mode=mode, format=fmt,
            gray_like=bool(np.array_equal(rgb[..., 0], rgb[..., 1]) and np.array_equal(rgb[..., 1], rgb[..., 2])),
            mean=float(rgb.mean()), std=float(rgb.std()), size_kb=p.stat().st_size / 1024,
            md5=_md5(p), dhash=dh, has_label_file=p.stem in label_files,
            n_boxes=len(boxes), boxes_json=json.dumps(boxes)))
    df = pd.DataFrame(rows)
    df['dhash'] = df['dhash'].astype(np.uint64)
    return df, unreadable


def drop_exact_duplicates(df):
    """Keep one file per identical-bytes group (files from Train/ are preferred, then by path).
    Groups whose copies carry different class labels are removed entirely (none expected)."""
    df = df.copy()
    df['_rank'] = (df.split_orig != 'train').astype(int)
    df = df.sort_values(['_rank', 'rel']).reset_index(drop=True)
    n_labels = df.groupby('md5').label.nunique()
    conflict_md5 = set(n_labels[n_labels > 1].index)
    df = df[~df.md5.isin(conflict_md5)]
    keep = ~df.md5.duplicated(keep='first')
    kept = df[keep][['md5', 'rel']].rename(columns={'rel': 'kept_rel'})
    log = df[~keep][['rel', 'label', 'split_orig', 'md5']].merge(kept, on='md5')
    out = df[keep].drop(columns='_rank').reset_index(drop=True)
    return out, log, len(conflict_md5)


def near_duplicate_groups(root, df, dhash_thr=2, corr_thr=0.95, thumb=64):
    """Group images that are visually near-identical although their bytes differ.
    A pair is linked only if (a) dHash Hamming distance <= dhash_thr AND
    (b) Pearson correlation of 64x64 grayscale thumbnails >= corr_thr.
    Linked pairs are merged into connected components (union-find)."""
    root = Path(root)
    H = df.dhash.values.astype(np.uint64)
    n = len(H)
    pop = np.array([bin(i).count('1') for i in range(256)], dtype=np.uint8)
    cand = []
    for i in range(0, n, 500):
        x = H[i:i + 500, None] ^ H[None, :]
        d = pop[x.view(np.uint8).reshape(x.shape[0], x.shape[1], 8)].sum(axis=2)
        ii, jj = np.where(d <= dhash_thr)
        cand += [(a + i, b, int(d[a, b])) for a, b in zip(ii, jj) if a + i < b]
    cache = {}
    def thumb_of(k):
        if k not in cache:
            im = Image.open(root / df.rel.iloc[k]).convert('L').resize((thumb, thumb))
            cache[k] = np.asarray(im, dtype=np.float32).ravel()
        return cache[k]
    parent = list(range(n))
    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]; a = parent[a]
        return a
    pairs = []
    for a, b, d in cand:
        x, y = thumb_of(a), thumb_of(b)
        c = float(np.corrcoef(x, y)[0, 1]) if x.std() > 0 and y.std() > 0 else float('nan')
        linked = c >= corr_thr
        pairs.append(dict(i=a, j=b, dhash_dist=d, corr=c, linked=linked,
                          label_i=df.label.iloc[a], label_j=df.label.iloc[b]))
        if linked:
            parent[find(a)] = find(b)
    group = np.array([find(k) for k in range(n)])
    _, group = np.unique(group, return_inverse=True)
    return group, pd.DataFrame(pairs)


def make_splits(df, seed=42, n_splits=7):
    """Stratified, group-aware split. With 7 folds: 1 fold test (~14.3%), 1 fold val (~14.3%),
    5 folds train (~71.4%), i.e. approximately 70/15/15. Images in the same group never cross splits."""
    from sklearn.model_selection import StratifiedGroupKFold
    df = df.copy()
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    fold = np.full(len(df), -1)
    for k, (_, te) in enumerate(sgkf.split(df, df.label_idx, df.group)):
        fold[te] = k
    df['split'] = np.where(fold == 0, 'test', np.where(fold == 1, 'val', 'train'))
    return df


def verify_splits(df, pairs=None):
    """Leakage checks on the final split table. Returns a dict of results."""
    res = {}
    res['n_images'] = len(df)
    res['counts'] = df.split.value_counts().to_dict()
    res['class_counts'] = pd.crosstab(df.split, df.label).to_dict('index')
    res['md5_shared_across_splits'] = int((df.groupby('md5').split.nunique() > 1).sum())
    res['groups_shared_across_splits'] = int((df.groupby('group').split.nunique() > 1).sum())
    res['rel_duplicated'] = int(df.rel.duplicated().sum())
    if pairs is not None and len(pairs):
        lk = pairs[pairs.linked]
        sp = df.split.values
        res['linked_near_dup_pairs'] = int(len(lk))
        res['linked_near_dup_pairs_across_splits'] = int((sp[lk.i.values] != sp[lk.j.values]).sum())
        res['linked_pairs_with_different_labels'] = int((lk.label_i != lk.label_j).sum())
    return res


def run_split_pipeline(root, seed=42, progress=lambda x, **k: x):
    """Full Stage 2 pipeline in one call: inventory -> exact de-duplication -> near-duplicate groups -> split.
    Deterministic for a given dataset, seed and library versions. Returns (splits_df, info_dict)."""
    inv, unreadable = build_inventory(root, progress)
    clean, dup_log, n_conflict = drop_exact_duplicates(inv)
    group, pairs = near_duplicate_groups(root, clean)
    clean['group'] = group
    splits = make_splits(clean, seed=seed)
    return splits, dict(n_found=len(inv), unreadable=unreadable, dup_log=dup_log, n_conflict=n_conflict, pairs=pairs)

In [ ]:
%%writefile /content/brain-tumor-project/src/shortcut_check.py
"""Shortcut diagnostic: how well can the class be predicted WITHOUT looking at anatomy?
Uses only image metadata and global pixel statistics. Fit on the train split, evaluated on the
validation split (the test split is not touched). A high score means a CNN could also exploit
such non-anatomical cues, so its accuracy must be interpreted with care."""
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

FEATURE_SETS = {
    'majority class (reference)': [],
    'image size only (width, height, aspect, is_512x512)': ['width', 'height', 'aspect', 'is_512'],
    'size + colour mode + file size': ['width', 'height', 'aspect', 'is_512', 'mode_code', 'gray_like', 'size_kb'],
    'size + mode + file size + global intensity (mean, std)': ['width', 'height', 'aspect', 'is_512', 'mode_code', 'gray_like', 'size_kb', 'mean', 'std'],
}

def run(df, seed=42):
    d = df.copy()
    d['aspect'] = d.width / d.height
    d['is_512'] = ((d.width == 512) & (d.height == 512)).astype(int)
    d['mode_code'] = d['mode'].astype('category').cat.codes
    d['gray_like'] = d.gray_like.astype(int)
    tr, va = d[d.split == 'train'], d[d.split == 'val']
    rows = []
    for name, feats in FEATURE_SETS.items():
        if feats:
            clf = RandomForestClassifier(n_estimators=300, random_state=seed, n_jobs=-1, class_weight='balanced')
            clf.fit(tr[feats], tr.label_idx); pred = clf.predict(va[feats])
        else:
            clf = DummyClassifier(strategy='most_frequent').fit(tr[['width']], tr.label_idx); pred = clf.predict(va[['width']])
        rows.append(dict(features=name, val_accuracy=accuracy_score(va.label_idx, pred),
                         val_macro_f1=f1_score(va.label_idx, pred, average='macro')))
        last_pred = pred
    return pd.DataFrame(rows), last_pred, va

In [ ]:
%%writefile /content/brain-tumor-project/src/preprocessing.py
"""Image preprocessing and augmentation.

Training preprocessing : pad to square -> resize -> mild random augmentation -> tensor -> normalise
Validation/test        : pad to square -> resize -> tensor -> normalise      (NO random operations)

Why pad instead of stretch: the images are not all square (for example 201x251), and stretching would
distort the shape of the head and of the tumour. Padding with black keeps the aspect ratio.
Why 3 channels: ImageNet-pretrained ResNets expect RGB input, so grayscale MRI is replicated to 3 channels
(done when the file is loaded, with PIL's convert('RGB')).
Why ImageNet mean/std: the pretrained weights were trained with this normalisation.
"""
from PIL import Image

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


class PadToSquare:
    """Pad a PIL image with a constant colour (default black) to a centred square."""
    def __init__(self, fill=0):
        self.fill = fill

    def __call__(self, img):
        w, h = img.size
        s = max(w, h)
        if w == h:
            return img
        canvas = Image.new(img.mode, (s, s), (self.fill,) * len(img.getbands()))
        canvas.paste(img, ((s - w) // 2, (s - h) // 2))
        return canvas


def get_transforms(train, img_size=224, hflip=False):
    """Return a torchvision transform. hflip is OFF by default (anatomical orientation, see report)."""
    from torchvision import transforms as T
    base = [PadToSquare(0), T.Resize((img_size, img_size))]
    tail = [T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)]
    if not train:
        return T.Compose(base + tail)
    aug = [
        T.RandomResizedCrop(img_size, scale=(0.85, 1.0), ratio=(0.95, 1.05)),   # moderate crop
        T.RandomAffine(degrees=10, translate=(0.05, 0.05)),                      # small rotation / shift
        T.ColorJitter(brightness=0.1, contrast=0.1),                             # slight intensity change
    ]
    if hflip:
        aug.insert(0, T.RandomHorizontalFlip(0.5))   # only for the separate ablation experiment
    return T.Compose(base + aug + tail)


def denormalize(t):
    """Tensor (3,H,W) normalised -> numpy (H,W,3) in [0,1] for plotting."""
    import torch
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1); std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (t * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

In [ ]:
%%writefile /content/brain-tumor-project/src/dataset.py
"""PyTorch Dataset and DataLoader construction."""
from pathlib import Path
import json
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from preprocessing import get_transforms


class MRIDataset(Dataset):
    """Reads rows of the split table. Images are opened lazily and converted to RGB."""
    def __init__(self, df, root, transform=None, return_meta=False):
        self.df = df.reset_index(drop=True)
        self.root = Path(root)
        self.transform = transform
        self.return_meta = return_meta

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(self.root / row.rel).convert('RGB')   # L / RGBA / P -> RGB
        if self.transform is not None:
            img = self.transform(img)
        label = int(row.label_idx)
        if self.return_meta:
            return img, label, dict(rel=row.rel, boxes=json.loads(row.boxes_json))
        return img, label


def _worker_init(worker_id):
    seed = torch.initial_seed() % 2**32
    np.random.seed(seed); import random; random.seed(seed)


def class_weights(train_df, n_classes=4):
    """Inverse-frequency weights (only used if the imbalance experiment is run)."""
    counts = np.bincount(train_df.label_idx.values, minlength=n_classes).astype(float)
    return torch.tensor(counts.sum() / (n_classes * counts), dtype=torch.float32)


def make_loaders(split_df, root, batch_size=32, img_size=224, hflip=False, num_workers=2, seed=42, sampler=None):
    """Return dict of train/val/test loaders. Only the training set uses random augmentation."""
    g = torch.Generator(); g.manual_seed(seed)
    loaders = {}
    for name in ['train', 'val', 'test']:
        part = split_df[split_df.split == name]
        ds = MRIDataset(part, root, get_transforms(train=(name == 'train'), img_size=img_size, hflip=hflip))
        loaders[name] = DataLoader(
            ds, batch_size=batch_size,
            shuffle=(name == 'train' and sampler is None),
            sampler=sampler if name == 'train' else None,
            num_workers=num_workers, pin_memory=torch.cuda.is_available(),
            worker_init_fn=_worker_init, generator=g if name == 'train' else None,
            drop_last=False)
    return loaders

In [ ]:
%%writefile /content/brain-tumor-project/src/models.py
"""Model definitions. Stage 3: baseline CNN. (ResNet builders are added in Stage 4.)"""
import torch
import torch.nn as nn


def conv_block(c_in, c_out):
    """Conv 3x3 -> BatchNorm -> ReLU -> MaxPool(2). Halves the spatial size."""
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(c_out),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(2))


class BaselineCNN(nn.Module):
    """Small CNN trained from scratch (no pretrained weights).
    5 conv blocks (32-64-128-256-256 filters) -> global average pooling -> dropout -> linear(4).
    For a 224x224 input the feature map is 7x7 before global pooling."""
    def __init__(self, n_classes=4, dropout=0.5):
        super().__init__()
        chans = [3, 32, 64, 128, 256, 256]
        self.features = nn.Sequential(*[conv_block(chans[i], chans[i + 1]) for i in range(5)])
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(dropout), nn.Linear(chans[-1], n_classes))
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')

    def forward(self, x):
        return self.classifier(self.pool(self.features(x)))


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

In [ ]:
%%writefile /content/brain-tumor-project/src/evaluate.py
"""Prediction, metrics, saving of results and the experiment table."""
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix,
                             precision_recall_fscore_support, roc_auc_score, f1_score)


def predict(model, loader, device, criterion=None):
    """Run the model on a loader WITHOUT shuffling. Returns labels, predictions, softmax probabilities,
    optional mean loss, and the relative file path of every image (same order)."""
    import torch
    model.eval()
    ys, logits, loss_sum, n = [], [], 0.0, 0
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device, non_blocking=True); y = y.to(device, non_blocking=True)
            out = model(x).float()
            if criterion is not None:
                loss_sum += criterion(out, y).item() * x.size(0)
            n += x.size(0); logits.append(out.cpu()); ys.append(y.cpu())
    probs = torch.softmax(torch.cat(logits), dim=1).numpy().astype(np.float64)
    y_true = torch.cat(ys).numpy()
    rels = loader.dataset.df.rel.values
    assert len(rels) == len(y_true), 'loader order / length mismatch'
    return dict(y_true=y_true, y_pred=probs.argmax(1), probs=probs,
                loss=(loss_sum / n) if criterion is not None else None, rels=rels)


def compute_metrics(y_true, probs, class_names):
    """All metrics are computed from the labels and probabilities passed in."""
    K = len(class_names); labels = list(range(K))
    y_pred = probs.argmax(1)
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
    mp, mr, mf, _ = precision_recall_fscore_support(y_true, y_pred, labels=labels, average='macro', zero_division=0)
    wp, wr, wf, _ = precision_recall_fscore_support(y_true, y_pred, labels=labels, average='weighted', zero_division=0)
    tp = np.diag(cm); fn = cm.sum(1) - tp; fp = cm.sum(0) - tp; tn = cm.sum() - tp - fn - fp
    sens = tp / np.maximum(tp + fn, 1)      # = recall of the class
    spec = tn / np.maximum(tn + fp, 1)      # one-vs-rest specificity
    pc = pd.DataFrame(dict(cls=class_names, precision=p, recall_sensitivity=r, specificity=spec, f1=f, support=s))
    P = probs / probs.sum(1, keepdims=True)
    try:
        auc_macro = roc_auc_score(y_true, P, multi_class='ovr', average='macro', labels=labels)
        auc_weighted = roc_auc_score(y_true, P, multi_class='ovr', average='weighted', labels=labels)
        auc_pc = [roc_auc_score((y_true == k).astype(int), P[:, k]) for k in labels]
    except ValueError:
        auc_macro = auc_weighted = float('nan'); auc_pc = [float('nan')] * K
    pc['roc_auc_ovr'] = auc_pc
    summary = dict(n=int(len(y_true)), accuracy=accuracy_score(y_true, y_pred),
                   balanced_accuracy=balanced_accuracy_score(y_true, y_pred),
                   precision_macro=mp, recall_macro=mr, f1_macro=mf,
                   precision_weighted=wp, recall_weighted=wr, f1_weighted=wf,
                   roc_auc_ovr_macro=auc_macro, roc_auc_ovr_weighted=auc_weighted)
    return dict(summary={k: float(v) if not isinstance(v, int) else v for k, v in summary.items()},
                per_class=pc, confusion_matrix=cm)


def bootstrap_ci(y_true, y_pred, n_boot=1000, seed=42, alpha=0.05):
    """Percentile bootstrap CI (resampling test images) for accuracy and macro-F1.
    It reflects test-set sampling noise only, not training randomness."""
    rng = np.random.default_rng(seed); n = len(y_true); acc, f1 = [], []
    for _ in range(n_boot):
        i = rng.integers(0, n, n)
        acc.append((y_true[i] == y_pred[i]).mean()); f1.append(f1_score(y_true[i], y_pred[i], average='macro'))
    q = [100 * alpha / 2, 100 * (1 - alpha / 2)]
    return dict(accuracy_ci=tuple(np.percentile(acc, q).round(4)), f1_macro_ci=tuple(np.percentile(f1, q).round(4)))


def evaluate_split(model, loader, device, class_names, tag, out_root, criterion=None, n_boot=1000, seed=42):
    """Predict, compute metrics, and save metrics / per-class table / confusion matrix / predictions."""
    out_root = Path(out_root)
    res = predict(model, loader, device, criterion)
    m = compute_metrics(res['y_true'], res['probs'], class_names)
    m['summary']['loss'] = res['loss']
    m['bootstrap_ci'] = bootstrap_ci(res['y_true'], res['y_pred'], n_boot, seed)
    save_results(tag, res, m, class_names, out_root)
    return m, res


def save_results(tag, res, m, class_names, out_root):
    out_root = Path(out_root)
    (out_root / 'metrics').mkdir(parents=True, exist_ok=True); (out_root / 'predictions').mkdir(parents=True, exist_ok=True)
    json.dump(dict(summary=m['summary'], bootstrap_ci=m.get('bootstrap_ci')), open(out_root / 'metrics' / f'{tag}_metrics.json', 'w'), indent=2, default=str)
    m['per_class'].to_csv(out_root / 'metrics' / f'{tag}_per_class.csv', index=False)
    pd.DataFrame(m['confusion_matrix'], index=class_names, columns=class_names).to_csv(out_root / 'metrics' / f'{tag}_confusion_matrix.csv')
    pred = pd.DataFrame(dict(rel=res['rel'] if 'rel' in res else res['rels'], y_true=res['y_true'], y_pred=res['y_pred']))
    pred['true_label'] = [class_names[i] for i in pred.y_true]; pred['pred_label'] = [class_names[i] for i in pred.y_pred]
    pred['correct'] = pred.y_true == pred.y_pred
    for k, c in enumerate(class_names):
        pred[f'prob_{c}'] = res['probs'][:, k]
    pred.to_csv(out_root / 'predictions' / f'{tag}_predictions.csv', index=False)


EXPERIMENT_COLUMNS = ['experiment_id', 'model', 'learning_rate', 'batch_size', 'optimizer', 'weight_decay', 'loss',
                      'max_epochs', 'best_epoch', 'epochs_run', 'finetune_strategy', 'augmentation',
                      'trainable_params', 'val_accuracy', 'val_macro_f1', 'test_accuracy', 'test_macro_f1',
                      'roc_auc_test_ovr_macro', 'train_time_s', 'seed']


def log_experiment(path, row):
    """Add or replace one row (by experiment_id) in the experiment table. Only measured values go in."""
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    df = pd.read_csv(path) if path.exists() else pd.DataFrame(columns=EXPERIMENT_COLUMNS)
    df = df[df.experiment_id != row['experiment_id']]
    df = pd.concat([df, pd.DataFrame([row]).reindex(columns=EXPERIMENT_COLUMNS)], ignore_index=True)
    df.to_csv(path, index=False)
    return df

In [ ]:
%%writefile /content/brain-tumor-project/src/train.py
"""Training loop: early stopping, best-model checkpointing, LR scheduling."""
import time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score
from evaluate import predict


def train_one_epoch(model, loader, criterion, optimizer, device, scaler, use_amp):
    """One pass over the training loader. The returned accuracy is a running average over AUGMENTED
    batches while the weights are changing, so it is not the same as accuracy on clean training images."""
    model.train()
    loss_sum, correct, n = 0.0, 0, 0
    for x, y in loader:
        x = x.to(device, non_blocking=True); y = y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, enabled=use_amp):
            out = model(x); loss = criterion(out, y)
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        loss_sum += loss.item() * x.size(0); correct += (out.argmax(1) == y).sum().item(); n += x.size(0)
    return loss_sum / n, correct / n


def fit(model, loaders, criterion, optimizer, device, epochs=40, patience=8, scheduler=None,
        ckpt_path='best.pt', monitor='val_macro_f1', use_amp=True, config=None, val_criterion=None):
    """Train with validation after every epoch.
    * The checkpoint is saved whenever the monitored VALIDATION metric improves (never training accuracy).
    * Training stops after `patience` epochs without improvement.
    * ReduceLROnPlateau schedulers are stepped with the validation loss.
    * At the end the best weights are loaded back into `model`.
    `config` must contain only plain Python values (it is stored inside the checkpoint)."""
    use_amp = bool(use_amp and device.type == 'cuda')
    try:
        scaler = torch.amp.GradScaler(device.type, enabled=use_amp)
    except (AttributeError, TypeError):
        scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    val_criterion = val_criterion or criterion
    ckpt_path = Path(ckpt_path); ckpt_path.parent.mkdir(parents=True, exist_ok=True)
    hist, best, best_epoch, wait, t_start = [], -np.inf, 0, 0, time.time()
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, loaders['train'], criterion, optimizer, device, scaler, use_amp)
        va = predict(model, loaders['val'], device, val_criterion)
        va_acc = float((va['y_true'] == va['y_pred']).mean())
        va_f1 = float(f1_score(va['y_true'], va['y_pred'], average='macro'))
        lr = optimizer.param_groups[0]['lr']
        if scheduler is not None:
            scheduler.step(va['loss'])
        row = dict(epoch=epoch, lr=lr, train_loss=tr_loss, train_acc=tr_acc, val_loss=va['loss'],
                   val_acc=va_acc, val_macro_f1=va_f1, seconds=time.time() - t0)
        hist.append(row)
        score = -row[monitor] if monitor.endswith('loss') else row[monitor]
        if score > best + 1e-4:
            best, best_epoch, wait = score, epoch, 0
            torch.save(dict(model_state=model.state_dict(), epoch=epoch, monitor=monitor,
                            monitor_value=float(row[monitor]), config=config or {}), ckpt_path)
            flag = ' *saved*'
        else:
            wait += 1; flag = ''
        print(f"Epoch {epoch:3d} | lr {lr:.2e} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
              f"val loss {va['loss']:.4f} acc {va_acc:.4f} macroF1 {va_f1:.4f} | {row['seconds']:.0f}s{flag}")
        if wait >= patience:
            print(f'Early stopping: no improvement in {monitor} for {patience} epochs.')
            break
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)   # our own file
    model.load_state_dict(ckpt['model_state'])
    summary = dict(best_epoch=best_epoch, best_monitor_value=float(ckpt['monitor_value']), monitor=monitor,
                   epochs_run=len(hist), stopped_early=len(hist) < epochs, train_time_s=time.time() - t_start,
                   amp=use_amp)
    return pd.DataFrame(hist), summary

In [ ]:
%%writefile /content/brain-tumor-project/src/visualization.py
"""Report-ready plots (matplotlib / seaborn only)."""
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc


def _save(fig, path):
    if path is not None:
        fig.savefig(path, dpi=200, bbox_inches='tight')


def plot_history(hist, title, path=None, best_epoch=None):
    """Loss curves (train/val) and accuracy curves (train/val), one panel each."""
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(hist.epoch, hist.train_loss, label='Training loss'); ax[0].plot(hist.epoch, hist.val_loss, label='Validation loss')
    ax[1].plot(hist.epoch, hist.train_acc, label='Training accuracy (augmented batches)'); ax[1].plot(hist.epoch, hist.val_acc, label='Validation accuracy')
    ax[1].plot(hist.epoch, hist.val_macro_f1, '--', label='Validation macro F1')
    for a, t, yl in zip(ax, ['Loss', 'Accuracy / F1'], ['Cross-entropy loss', 'Score']):
        if best_epoch: a.axvline(best_epoch, color='gray', ls=':', label=f'Best epoch ({best_epoch})')
        a.set_xlabel('Epoch'); a.set_ylabel(yl); a.set_title(t); a.grid(alpha=.3); a.legend(fontsize=8)
    fig.suptitle(title); fig.tight_layout(); _save(fig, path); return fig


def plot_confusion_matrix(cm, class_names, title, path=None):
    """Counts (left) and row-normalised percentages = recall per class (right)."""
    cm = np.asarray(cm); norm = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    fig, ax = plt.subplots(1, 2, figsize=(11, 4.4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=ax[0], cbar=False)
    sns.heatmap(norm * 100, annot=True, fmt='.1f', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=ax[1], cbar=False, vmin=0, vmax=100)
    ax[0].set_title('Counts'); ax[1].set_title('Row-normalised (% of true class)')
    for a in ax: a.set_xlabel('Predicted'); a.set_ylabel('True'); a.tick_params(axis='x', rotation=20)
    fig.suptitle(title); fig.tight_layout(); _save(fig, path); return fig


def plot_roc(y_true, probs, class_names, title, path=None):
    """One-vs-rest ROC curve for every class."""
    fig, ax = plt.subplots(figsize=(5.5, 5))
    for k, c in enumerate(class_names):
        fpr, tpr, _ = roc_curve((y_true == k).astype(int), probs[:, k])
        ax.plot(fpr, tpr, label=f'{c} (AUC = {auc(fpr, tpr):.3f})')
    ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Chance')
    ax.set_xlabel('False positive rate'); ax.set_ylabel('True positive rate'); ax.set_title(title); ax.legend(fontsize=8, loc='lower right'); ax.grid(alpha=.3)
    fig.tight_layout(); _save(fig, path); return fig


def plot_per_class(per_class, title, path=None):
    """Grouped bars: precision, recall, F1 for every class."""
    d = per_class.melt(id_vars='cls', value_vars=['precision', 'recall_sensitivity', 'f1'], var_name='metric', value_name='value')
    d['metric'] = d.metric.replace({'recall_sensitivity': 'recall'})
    fig, ax = plt.subplots(figsize=(7.5, 4))
    sns.barplot(data=d, x='cls', y='value', hue='metric', ax=ax)
    for c in ax.containers: ax.bar_label(c, fmt='%.2f', fontsize=7)
    ax.set_ylim(0, 1.08); ax.set_xlabel('Class'); ax.set_ylabel('Score'); ax.set_title(title); ax.legend(title='')
    fig.tight_layout(); _save(fig, path); return fig

## 2. Imports, seed, device

In [ ]:
import sys, json, shutil, warnings, os, subprocess, zipfile
sys.path.insert(0, str(PROJECT/'src'))
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch, torch.nn as nn
from collections import Counter
from tqdm.auto import tqdm
import data_prep as dp, shortcut_check as sc
from utils import set_seed, env_info
from dataset import make_loaders, class_weights
from models import BaselineCNN, count_params
from train import fit
from evaluate import evaluate_split, log_experiment
import visualization as vz

SEED = 42
set_seed(SEED)
ENV = env_info(SEED); print(json.dumps(ENV, indent=2))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type != 'cuda':
    print('WARNING: no GPU found. Runtime > Change runtime type > GPU. Training on CPU will be very slow.')
CLASS_NAMES = dp.CLASS_NAMES

## 3. Optional: keep results in Google Drive
Colab sessions are temporary. Set `USE_DRIVE = True` to restore earlier results (including the split table) from Drive at the start and to back up results/models at the end.

In [ ]:
USE_DRIVE = False
BACKUP = Path('/content/drive/MyDrive/brain-tumor-project')
def sync_to_drive():
    if not USE_DRIVE: return
    for sub in ['results','models']:
        shutil.copytree(PROJECT/sub, BACKUP/sub, dirs_exist_ok=True)
    print('Backed up to', BACKUP)
if USE_DRIVE:
    from google.colab import drive; drive.mount('/content/drive')
    for sub in ['results','models']:
        if (BACKUP/sub).exists(): shutil.copytree(BACKUP/sub, PROJECT/sub, dirs_exist_ok=True); print('Restored', sub)

## 4. Dataset setup (same options as Stage 2)

In [ ]:
DATA_SOURCE = 'kaggle'                                         # 'kaggle' or 'drive'
DRIVE_ZIP = '/content/drive/MyDrive/brain_tumor_dataset.zip'   # only used if DATA_SOURCE == 'drive'
SLUG = 'ahmedsorour1/mri-for-brain-tumor-with-bounding-boxes'
RAW = PROJECT/'data'/'raw'; RAW.mkdir(exist_ok=True)
if not any(p.is_file() for p in RAW.rglob('*')):
    if DATA_SOURCE == 'drive':
        from google.colab import drive; drive.mount('/content/drive')
        assert Path(DRIVE_ZIP).exists(), f'Not found: {DRIVE_ZIP}'
        with zipfile.ZipFile(DRIVE_ZIP) as f: f.extractall(RAW)
    else:
        if not (os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY')):
            try:
                from google.colab import userdata
                os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME'); os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
            except Exception:
                from google.colab import files
                print('Upload kaggle.json'); up = files.upload()
                kd = Path.home()/'.kaggle'; kd.mkdir(exist_ok=True); (kd/'kaggle.json').write_bytes(up[list(up)[0]]); os.chmod(kd/'kaggle.json', 0o600)
        subprocess.run(['pip','-q','install','kaggle'], check=True)
        subprocess.run(['kaggle','datasets','download','-d',SLUG,'-p',str(RAW),'--unzip'], check=True)
ROOT = dp.find_dataset_root(RAW); print('Dataset root:', ROOT)

## 5. Load the Stage 2 split table (locked)
All experiments must use exactly the split produced by notebook 02, so it is **not** rebuilt here.
If `splits.csv` is not present in this session, you are asked to upload it (it is inside `preprocessing_results.zip`, folder `metrics`).
The file is verified against the expected counts and a content hash.

In [ ]:
import hashlib
SPLITS_CSV = PROJECT/'results/metrics/splits.csv'
EXPECTED_COUNTS = {'train': 2806, 'val': 563, 'test': 544}
EXPECTED_HASH = '5267697ab570ec7b2b9c4bbde7b21da3427cff24b9ced5c00ff4bd9483c146cb'
if not SPLITS_CSV.exists():
    from google.colab import files
    print('Upload splits.csv (from preprocessing_results.zip > metrics)')
    up = files.upload(); SPLITS_CSV.write_bytes(up[list(up)[0]])
splits = pd.read_csv(SPLITS_CSV)
got_hash = hashlib.sha256('\n'.join(f'{r}|{sp}' for r, sp in sorted(zip(splits.rel, splits.split))).encode()).hexdigest()
assert splits.split.value_counts().to_dict() == EXPECTED_COUNTS, ('Unexpected split sizes', splits.split.value_counts().to_dict())
assert got_hash == EXPECTED_HASH, 'This splits.csv is not the one produced by notebook 02. Upload the correct file.'
missing = [r for r in splits.rel if not (ROOT/r).exists()]
assert not missing, f'{len(missing)} files in the split table are missing, e.g. {missing[:3]}'
chk = dp.verify_splits(splits); assert chk['md5_shared_across_splits'] == 0 and chk['groups_shared_across_splits'] == 0
print('Split table verified: sizes and content match notebook 02; no identical files or duplicate groups across splits.')
print(pd.crosstab(splits.split, splits.label, margins=True))
train_df = splits[splits.split == 'train']
print('Imbalance ratio in train (largest/smallest class): %.2f' % (train_df.label.value_counts().max() / train_df.label.value_counts().min()))

## 6. Data loaders and model sanity checks

In [ ]:
BATCH_SIZE, IMG_SIZE = 32, 224
loaders = make_loaders(splits, ROOT, batch_size=BATCH_SIZE, img_size=IMG_SIZE, hflip=False, num_workers=2, seed=SEED)
for k, dl in loaders.items(): print(f'{k:5s}: {len(dl.dataset)} images, {len(dl)} batches')

m = BaselineCNN().to(device)
total, trainable = count_params(m); print(f'\nBaselineCNN parameters: {total:,} (trainable {trainable:,})')
print(m)
xb, yb = next(iter(loaders['train'])); xb, yb = xb.to(device), yb.to(device)
print('\nOutput shape for one batch:', tuple(m(xb).shape))
crit = nn.CrossEntropyLoss()
with torch.no_grad(): m.eval(); print('Loss of the untrained model: %.3f (a model guessing uniformly over 4 classes gives ln(4) = %.3f)' % (crit(m(xb), yb).item(), np.log(4)))

# can the model memorise ONE batch? If not, there is a bug in the model or the pipeline
set_seed(SEED); m = BaselineCNN().to(device); opt = torch.optim.Adam(m.parameters(), 1e-3); m.train(); losses = []
for i in range(40):
    opt.zero_grad(); l = crit(m(xb), yb); l.backward(); opt.step(); losses.append(l.item())
print('Single-batch overfit test: loss %.3f -> %.3f' % (losses[0], losses[-1]))
assert losses[-1] < losses[0], 'Loss did not decrease: check the model and the data pipeline.'
del m, opt

## 7. Experiment runner
Each experiment: seed -> train with early stopping on validation macro F1 -> load the best checkpoint -> evaluate on validation and test -> save curves, metrics, confusion matrix, ROC, per-class plot, predictions -> add a row to the experiment table.
The test results are recorded for reporting only; no choice is made from them.

In [ ]:
AUG_DESC = 'RandomResizedCrop(0.85-1.0), Affine(10deg, 5% shift), ColorJitter(0.1,0.1); no hflip'
EXP_TABLE = PROJECT/'results/metrics/experiment_table.csv'
CLASS_W = class_weights(train_df).to(device)

def run_experiment(exp_id, weighted=False, lr=1e-3, weight_decay=0.0, epochs=40, patience=8, dropout=0.5):
    print('='*80); print(f'Experiment {exp_id} | class-weighted loss: {weighted} | lr {lr} | wd {weight_decay} | batch {BATCH_SIZE} | max epochs {epochs} | patience {patience}')
    set_seed(SEED)
    model = BaselineCNN(dropout=dropout).to(device)
    criterion = nn.CrossEntropyLoss(weight=CLASS_W if weighted else None)
    val_criterion = nn.CrossEntropyLoss()                     # unweighted, so validation loss is comparable across experiments
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
    cfg = dict(experiment_id=exp_id, model='BaselineCNN', lr=lr, weight_decay=weight_decay, batch_size=BATCH_SIZE, epochs=epochs,
               patience=patience, dropout=dropout, weighted_loss=bool(weighted), seed=SEED, img_size=IMG_SIZE, augmentation=AUG_DESC)
    ckpt = PROJECT/'models'/f'{exp_id}_best.pt'
    hist, summ = fit(model, loaders, criterion, optimizer, device, epochs=epochs, patience=patience, scheduler=scheduler,
                     ckpt_path=ckpt, monitor='val_macro_f1', use_amp=True, config=cfg, val_criterion=val_criterion)
    hist.to_csv(PROJECT/f'results/metrics/{exp_id}_history.csv', index=False)
    print(f"\nBest epoch: {summ['best_epoch']} (val macro F1 {summ['best_monitor_value']:.4f}) | epochs run: {summ['epochs_run']} | time {summ['train_time_s']/60:.1f} min")

    R = PROJECT/'results'
    m_val, _ = evaluate_split(model, loaders['val'], device, CLASS_NAMES, f'{exp_id}_val', R, val_criterion)
    m_test, res_test = evaluate_split(model, loaders['test'], device, CLASS_NAMES, f'{exp_id}_test', R, val_criterion)

    vz.plot_history(hist, f'{exp_id}: training history', R/f'figures/{exp_id}_curves.png', best_epoch=summ['best_epoch']); plt.show()
    vz.plot_confusion_matrix(m_test['confusion_matrix'], CLASS_NAMES, f'{exp_id}: test confusion matrix', R/f'figures/{exp_id}_confusion_matrix_test.png'); plt.show()
    vz.plot_roc(res_test['y_true'], res_test['probs'], CLASS_NAMES, f'{exp_id}: test ROC (one-vs-rest)', R/f'figures/{exp_id}_roc_test.png'); plt.show()
    vz.plot_per_class(m_test['per_class'], f'{exp_id}: per-class test metrics', R/f'figures/{exp_id}_per_class_test.png'); plt.show()

    row = dict(experiment_id=exp_id, model='BaselineCNN (scratch)', learning_rate=lr, batch_size=BATCH_SIZE, optimizer='Adam',
               weight_decay=weight_decay, loss='CrossEntropy' + (' (class-weighted)' if weighted else ''), max_epochs=epochs,
               best_epoch=summ['best_epoch'], epochs_run=summ['epochs_run'], finetune_strategy='none (trained from scratch)',
               augmentation=AUG_DESC, trainable_params=count_params(model)[1],
               val_accuracy=m_val['summary']['accuracy'], val_macro_f1=m_val['summary']['f1_macro'],
               test_accuracy=m_test['summary']['accuracy'], test_macro_f1=m_test['summary']['f1_macro'],
               roc_auc_test_ovr_macro=m_test['summary']['roc_auc_ovr_macro'], train_time_s=round(summ['train_time_s'], 1), seed=SEED)
    log_experiment(EXP_TABLE, row)
    return dict(model=model, hist=hist, summ=summ, val=m_val, test=m_test, res_test=res_test)

def report(r, name):
    for split in ['val', 'test']:
        s = r[split]['summary']
        print(f"\n{name} | {split.upper()} (n={s['n']}): accuracy {s['accuracy']:.4f} | balanced acc {s['balanced_accuracy']:.4f} | "
              f"macro P/R/F1 {s['precision_macro']:.4f}/{s['recall_macro']:.4f}/{s['f1_macro']:.4f} | weighted F1 {s['f1_weighted']:.4f} | ROC-AUC (OvR macro) {s['roc_auc_ovr_macro']:.4f}")
        if split == 'test': print('   bootstrap 95% CI  accuracy:', r[split]['bootstrap_ci']['accuracy_ci'], ' macro F1:', r[split]['bootstrap_ci']['f1_macro_ci'])
    print('\nPer-class TEST metrics:'); print(r['test']['per_class'].round(4).to_string(index=False))

## 8. Experiment B1: baseline CNN, plain cross-entropy

In [ ]:
B1 = run_experiment('B1', weighted=False)
report(B1, 'B1')

## 9. Reference: how well do non-anatomical cues alone do? (from Stage 2)
The metadata-only model was scored on the same validation split. Compare it with the CNN's validation result.

In [ ]:
short, _, _ = sc.run(splits, seed=SEED)
print(short.round(3).to_string(index=False))
s = B1['val']['summary']
print(f"\nB1 CNN, validation: accuracy {s['accuracy']:.3f}, macro F1 {s['f1_macro']:.3f}")

## 10. Experiment B2 (optional): class-weighted cross-entropy
Same model, seed and settings as B1. Only the training loss changes (inverse-frequency class weights). Validation loss is still unweighted.

In [ ]:
RUN_B2 = True
if RUN_B2:
    B2 = run_experiment('B2', weighted=True)
    report(B2, 'B2')
    cmp = pd.DataFrame({'B1 (plain CE)': B1['val']['summary'], 'B2 (weighted CE)': B2['val']['summary']}).loc[['accuracy','balanced_accuracy','f1_macro','roc_auc_ovr_macro']]
    print('\nValidation comparison (selection uses validation only):'); print(cmp.round(4))
    print('\nValidation per-class recall:')
    print(pd.DataFrame({'B1': B1['val']['per_class'].set_index('cls').recall_sensitivity, 'B2': B2['val']['per_class'].set_index('cls').recall_sensitivity}).round(4))

## 11. Experiment table and backup

In [ ]:
table = pd.read_csv(EXP_TABLE); display(table.round(4))
sync_to_drive()
print('\nSaved files:'); 
for p in sorted((PROJECT/'results').rglob('*')):
    if p.is_file() and p.stem.startswith('B'): print(' ', p.relative_to(PROJECT))
print('Models:', [p.name for p in (PROJECT/'models').glob('*.pt')])

## What to send back
The printed output of sections 5 to 11 (especially the epoch log, the `report` blocks, the validation comparison and the experiment table), plus the figures `B1_curves.png`, `B1_confusion_matrix_test.png`, `B1_roc_test.png`. Stage 4 (ResNet18) will be written after that, using your actual baseline numbers.